In [1]:
from pathlib import Path
from collections import Counter

docs_dir = Path("documents")
files = [f for f in docs_dir.rglob("*") if f.is_file()]

print("Total files:", len(files))
print("File types:", Counter(f.suffix.lower() for f in files))
print("Folders:", Counter(f.parent.name for f in files))

Total files: 183
File types: Counter({'.pdf': 166, '': 13, '.ds_store': 3, '.zip': 1})
Folders: Counter({'product_catalog': 126, 'internal_docs_by_area': 9, 'HR_policies': 8, 'Customer_support': 6, 'Deliveries': 6, 'Finance': 6, 'Legal_Terms': 6, 'Sales': 6, 'documents': 5, 'IT': 4, '__MACOSX': 1})


In [2]:
#extracting the only pdf files
pdfs = [f for f in files if f.suffix.lower() == ".pdf" and "__MACOSX" not in str(f)]
print("Real PDFs:", len(pdfs))

Real PDFs: 83


In [3]:
%pip install pandas pypdf

Note: you may need to restart the kernel to use updated packages.


In [5]:
#Read all the text into a table
import pandas as pd
from pypdf import PdfReader

rows = []
for f in pdfs:
    reader = PdfReader(f)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    rows.append({
        "file": f.name,
        "folder": f.parent.name,
        "pages": len(reader.pages),
        "characters": len(text),
        "text": text,
    })

df = pd.DataFrame(rows)
df[["file", "folder", "pages", "characters"]]

,file,folder,pages,characters
0,IronStore_Customer_Support_Escalation_Procedur...,Customer_support,7,12094
1,IronStore_Customer_Support_Knowledge_Base_Styl...,Customer_support,7,11810
2,IronStore_Refund_and_Complaint_Handling_Policy...,Customer_support,7,13552
3,IronStore_Returns_and_Reverse_Logistics_Proced...,Deliveries,6,12402
4,IronStore_Shipping_and_Delivery_SLA_Policy.pdf,Deliveries,6,11734
...,...,...,...,...
78,IronStore_Product_Catalog_Wall_Art_and_Decor.pdf,product_catalog,30,30808
79,IronStore_Product_Catalog_Water_Sports_and_Swi...,product_catalog,33,38403
80,IronStore_Product_Catalog_Wearable_Tech.pdf,product_catalog,32,24690
81,IronStore_Product_Catalog_Winter_Sports_Equipm...,product_catalog,24,25892


In [8]:
#finding how big is the data
print("Total pages:", df["pages"].sum())
print(df.groupby("folder")[["pages", "characters"]].sum())
print("\nEmpty or almost empty docs:")
print(df[df["characters"] < 200][["file", "characters"]])

Total pages: 2308
                  pages  characters
folder                             
Customer_support     21       37456
Deliveries           19       37657
Finance              18       36782
HR_policies          19       38093
IT                   12       27066
Legal_Terms          19       37156
Sales                19       36181
product_catalog    2181     2190839

Empty or almost empty docs:
Empty DataFrame
Columns: [file, characters]
Index: []


In [6]:
for i in [0, 30, 70]:
    print("=====", df.loc[i, "file"])
    print(df.loc[i, "text"][:600])
    print()

===== IronStore_Customer_Support_Escalation_Procedures.pdf
Customer Support Escalation Procedures
Owning area: Customer_support
Last reviewed: August 2026
Document ID: CS-POL-014
Version: 3.2
Applies to: IronStore customer support operations across Europe
1. Purpose and Overview
This procedure defines how IronStore identifies, manages, and escalates customer contacts that cannot be resolved
through standard frontline support. It is intended to ensure consistent decisions, timely ownership, and appropriate
protection of customer, payment, product, and company information.
Escalation is required when a case involves material customer impact, operational

===== IronStore_Product_Catalog_Books_and_Stationery.pdf
IronStore Product Catalog
Books & Stationery
Last reviewed in August 2026
Internal catalogue reference: IS-CAT-BKS-0826
Market: European Union and EEA
Currency: EUR, including applicable standard VAT unless otherwise stated
Availability definitions:
   In stock: Normally dispatche

In [10]:
#Extract the metadata with regex (NLP!)
import re

def find(pattern, text):
    m = re.search(pattern, text)
    return m.group(1).strip() if m else None

df["owning_area"]   = df["text"].apply(lambda t: find(r"Owning area:\s*(.+)", t))
df["version"]       = df["text"].apply(lambda t: find(r"Version:\s*([\d.]+)", t))
df["last_reviewed"] = df["text"].apply(lambda t: find(r"Last reviewed(?: in|:)\s*([A-Za-z]+ \d{4})", t))

df[["file", "owning_area", "version", "last_reviewed"]]

,file,owning_area,version,last_reviewed
0,IronStore_Customer_Support_Escalation_Procedur...,Customer_support,3.2,August 2026
1,IronStore_Customer_Support_Knowledge_Base_Styl...,Customer_support,NaN,August 2026
2,IronStore_Refund_and_Complaint_Handling_Policy...,Customer_support,3.2,August 2026
3,IronStore_Returns_and_Reverse_Logistics_Proced...,Deliveries,3.2,August 2026
4,IronStore_Shipping_and_Delivery_SLA_Policy.pdf,Deliveries,NaN,August 2026
...,...,...,...,...
78,IronStore_Product_Catalog_Wall_Art_and_Decor.pdf,NaN,NaN,August 2026
79,IronStore_Product_Catalog_Water_Sports_and_Swi...,NaN,NaN,August 2026
80,IronStore_Product_Catalog_Wearable_Tech.pdf,NaN,NaN,August 2026
81,IronStore_Product_Catalog_Winter_Sports_Equipm...,NaN,NaN,August 2026


In [11]:
#find outdated documents
print(df["last_reviewed"].value_counts(dropna=False))
df[df["file"].str.contains("old|2021|archive", case=False)][["file", "last_reviewed"]]

last_reviewed
August 2026      82
December 2021     1
Name: count, dtype: int64


,file,last_reviewed
20,IronStore_Old_Product_Catalog_2021.pdf,December 2021


In [12]:
#find duplicate and conflicts
from collections import defaultdict

sku_files = defaultdict(set)
for _, row in df.iterrows():
    for sku in re.findall(r"\bIS-[A-Z]+-[A-Z0-9-]+", row["text"]):
        sku_files[sku].add(row["file"])

repeated = {s: f for s, f in sku_files.items() if len(f) > 1}
print("Total SKUs:", len(sku_files))
print("SKUs in more than one file:", len(repeated))
for sku, fls in repeated.items():
    print(sku, "->", fls)

Total SKUs: 1448
SKUs in more than one file: 4
IS-BED-001 -> {'IronStore_Product_Catalog_Home_Essentials.pdf', 'IronStore_Product_Catalog_Bedding_and_Linens.pdf'}
IS-BED-002 -> {'IronStore_Product_Catalog_Home_Essentials.pdf', 'IronStore_Product_Catalog_Bedding_and_Linens.pdf'}
IS-BED-003 -> {'IronStore_Product_Catalog_Home_Essentials.pdf', 'IronStore_Product_Catalog_Bedding_and_Linens.pdf'}
IS-BED-004 -> {'IronStore_Product_Catalog_Home_Essentials.pdf', 'IronStore_Product_Catalog_Bedding_and_Linens.pdf'}


In [13]:
sku = list(repeated)[0]
for _, row in df.iterrows():
    i = row["text"].find(sku)
    if i >= 0:
        print("##", row["file"])
        print(row["text"][i:i+300], "\n")

## IronStore_Product_Catalog_Bedding_and_Linens.pdf
IS-BED-001 — Nordhaven Percale Bedding Set, White, 140 × 200 cm
Category: Bedding & Linens > Duvet Cover Sets
Price: €49.90
Availability: In stock
Warehouse location: Rotterdam, Netherlands — WH-NL01
Short description:
Three-piece duvet cover set made from combed cotton percale. Intended for single  

## IronStore_Product_Catalog_Home_Essentials.pdf
IS-BED-001 — Somnia Cotton Percale Duvet Set

Category: Home Essentials > Bedroom > Bedding
Short description: 100% cotton percale duvet-cover set with matching pillowcases.
Price: €49.90
Availability: In stock
Warehouse location: NL-RTM
Key specifications:
   Size: Double, 200 × 200 cm
  Pillowca 



In [14]:
print("HTML leftovers:", df["text"].str.count("<div").sum())
print("Broken characters ■:", df["text"].str.count("■").sum())
print(df[df["text"].str.contains("■")]["file"].head())

HTML leftovers: 22
Broken characters ■: 287
20               IronStore_Old_Product_Catalog_2021.pdf
21    IronStore_Product_Catalog_Art_and_Craft_Suppli...
22    IronStore_Product_Catalog_Audio_and_Headphones...
24       IronStore_Product_Catalog_Baby_and_Nursery.pdf
26    IronStore_Product_Catalog_Bathroom_Accessories...
Name: file, dtype: str


## Findings
- 83 PDFs: 20 policies (7 departments) + 63 product catalogs
- Every policy has a header: Owning area, Version, Last reviewed
- Outdated: Old_Product_Catalog_2021 (reviewed Dec 2021; all others Aug 2026)
- Conflicts: 4 SKUs used for different products in different catalogs
- Inconsistent formats: "Last reviewed in" vs "Last reviewed:"
- Noise: HTML tags, broken characters (■), Mac junk files
